In [13]:
import cv2
import numpy as np
import os

# Global variables for drawing the line
drawing = False
line_start = None
line_end = None
scale = None

def draw_line(event, x, y, flags, param):
    global drawing, line_start, line_end
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        line_start = (x, y)
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        line_end = (x, y)
        cv2.line(param, line_start, line_end, (0, 255, 0), 2)
        cv2.imshow("Image", param)

def calculate_scale(image, real_distance):
    global line_start, line_end, scale
    print("Draw a line on the image and press any key when done.")
    cv2.imshow("Image", image)
    cv2.setMouseCallback("Image", draw_line, param=image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    if line_start and line_end:
        pixel_distance = np.sqrt((line_end[0] - line_start[0])**2 + (line_end[1] - line_start[1])**2)
        scale = real_distance / pixel_distance
        print(f"Scale calculated: {scale} units per pixel.")
    else:
        print("No line was drawn. Exiting.")
        exit()

def process_image(image_path, output_folder, real_distance):
    global scale
    image = cv2.imread(image_path)
    if image is None:
        print(f"Failed to load image: {image_path}")
        return

    # Prompt the user to draw a line and calculate the scale
    if scale is None:
        calculate_scale(image.copy(), real_distance)

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Detect edges using Canny edge detection
    edges = cv2.Canny(gray, 100, 200)

    # Extract edge points
    edge_points = np.argwhere(edges > 0)

    # Convert pixel coordinates to real-world coordinates
    real_world_points = [(x * scale, y * scale) for x, y in edge_points]

    # Fit a polynomial curve to the real-world points
    real_world_points = np.array(real_world_points)
    x_coords = real_world_points[:, 1]  # Real-world x-coordinates
    y_coords = real_world_points[:, 0]  # Real-world y-coordinates
    polynomial_coefficients = np.polyfit(x_coords, y_coords, deg=2)  # Fit a quadratic curve

    print(f"Polynomial coefficients for {os.path.basename(image_path)}: {polynomial_coefficients}")

    # Save the output image with edges drawn
    output_image = np.zeros_like(image)
    for x, y in edge_points:
        output_image[x, y] = (255, 255, 255)  # Draw edges in white

    output_path = os.path.join(output_folder, f"processed_{os.path.basename(image_path)}")
    cv2.imwrite(output_path, output_image)
    print(f"Processed and saved: {output_path}")

def process_folder(folder_path, output_folder, real_distance):
    os.makedirs(output_folder, exist_ok=True)
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            process_image(os.path.join(folder_path, filename), output_folder, real_distance)

In [ ]:
# Example usage
folder_path = 'test-cans/8.16x4.83/'  # Replace with your folder path
single_image_path = 'test-cans/8.16x4.83/pepsi.jpg'  # Replace with your single image path
output_folder = 'output-images/'  # Replace with your output folder path
real_distance = 10.0  # Replace with the real-world distance for the drawn line

# Process a folder of images
#process_folder(folder_path, output_folder, real_distance)

# Process a single image
process_image(single_image_path, output_folder, real_distance)